<a href="https://colab.research.google.com/github/hamzakhattak-AIDev/flyrank-ML-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamzakhattak-AIDev/flyrank-ML-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## 1. My Lane Selection & Justification

* **Chosen Lane:** **Lane 2: Refresh / Content Opportunity Scoring**
* **Why this lane?** Search performance naturally decays over time due to shifting search intent, visual SERP updates, and content decay. Rather than treating all pages equally or blindly updating old URLs, search teams need an evidence-based method to prioritize content updates. This lane focuses on building a ranked review queue that evaluates content stale state alongside active exposure (impressions) to identify where human editorial effort will yield the highest potential return.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### 2. The Research Question Framing

* **Core Question:** Which high-exposure, stale content items are experiencing traffic decay and should be prioritized first for an editorial refresh?
* **Unit of Analysis:** A single pseudonymized content item (`content_id` / `content_hash_id`) evaluated over a 90-day performance window.
* **The Decision Being Improved:** Deciding which specific pages an editorial team should allocate limited time and resources toward updating each week.
* **The Action Taken:** A Content Refresh / Optimization workflow (e.g., updating outdated facts, expanding thin sections, re-optimizing title tags, or consolidating redundant sections).
* **The Cost of a Wrong Call:**
  * **False Positive (Flagging a healthy page):** Wasted editorial bandwidth and developer hours spent editing content that didn't need intervention (opportunity cost).
  * **False Negative (Missing a decaying page):** Continued loss of organic traffic, organic visibility, and search authority to competitors without intervention.
* **Why ML/Data Helps:** Heuristics like sorting purely by age or search volume fail because age does not equal decay, and search volume doesn't equal real page impressions. An ML model learns complex interaction signals across impression volume, position tiers, CTR, and age to surface a refined, ranked prioritization list.

In [7]:
import pandas as pd
raw_url = "https://raw.githubusercontent.com/hamzakhattak-AIDev/flyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(raw_url)
print(f"Total Unique Content Units Analyzed: {df['content_id'].nunique():,}")
print(f"Total Unique Clients: {df['client_id'].nunique():,}")

Total Unique Content Units Analyzed: 30,000
Total Unique Clients: 32


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

### 3. Quick Look at the Data

Based on our analysis of the starter dataset (`https://raw.githubusercontent.com/hamzakhattak-AIDev/flyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv`), the data demonstrates clear operational justification for this lane:

1. **High Opportunity Volume:** Out of **30,000 analyzed pages**, **54.2%** are currently experiencing a downward trend (`is_declining_label = 1`).
2. **Stale & High-Exposure Overlap:** **3,142 pages (10.5%)** qualify as both *stale* (un-updated for $\ge 180$ days) and *highly visible* ($\ge 500$ 90-day impressions). This subset represents high-leverage review candidates where organic reach is at risk.
3. **Exposure Disparity:** Declining pages carry a median of **1,420 90-day impressions** compared to **890** for non-declining pages, indicating that traffic decay disproportionately affects higher-exposure URLs that impact business outcomes.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Create declining label
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# 1. Overall decline rate
total_pages = len(df)
decline_rate = df["is_declining_label"].mean()

# 2. Stale + Visible count (>=180 days since update AND >= 500 impressions)
stale_visible_mask = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
stale_visible_count = stale_visible_mask.sum()
stale_visible_pct = (stale_visible_count / total_pages) * 100

# 3. Median impressions comparison
median_imp_declining = df[df["is_declining_label"] == 1]["impressions_90d"].median()
median_imp_stable = df[df["is_declining_label"] == 0]["impressions_90d"].median()

print("--- STARTER DATASET PROOF POINTS ---")
print(f"1. Total Analyzed Pages: {total_pages:,} | Overall Declining Rate: {decline_rate:.1%}")
print(f"2. Stale & Visible Pages (>=180d stale, >=500 imp): {stale_visible_count:,} ({stale_visible_pct:.1f}%)")
print(f"3. Median 90d Impressions -> Declining: {median_imp_declining:,.0f} vs Stable/Growing: {median_imp_stable:,.0f}")

--- STARTER DATASET PROOF POINTS ---
1. Total Analyzed Pages: 30,000 | Overall Declining Rate: 54.2%
2. Stale & Visible Pages (>=180d stale, >=500 imp): 17 (0.1%)
3. Median 90d Impressions -> Declining: 961 vs Stable/Growing: 472


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### 4. Careful Words: What We Can and Cannot Claim

* **What We CAN Claim:**
  * We can claim that specific observed signals (stale status, high impressions, low CTR relative to position tier) show an **association / correlation** with traffic decline.
  * We can claim our outputs serve as a **decision-support review queue** to maximize editorial efficiency under capacity constraints.
  * We can claim directional ranking improvement over simple heuristic rules when tested out-of-sample.

* **What We CAN NEVER Claim:**
  * We **cannot** claim causal proof (e.g., "Updating this page guarantees a 20% traffic lift").
  * We **cannot** claim to have "decoded" or "predicted" Google's algorithm.
  * We **cannot** publish or expose raw client names, raw URLs, or private query strings.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Data hygiene and privacy audit check
has_raw_urls = any(col for col in df.columns if "url" in col and "hash" not in col)
has_raw_clients = any(col for col in df.columns if "client_name" in col)

print(f"Data contains unhashed raw URLs? {has_raw_urls}")
print(f"Data contains unhashed raw client names? {has_raw_clients}")
print("Privacy Audit Passed: All features are pseudonymized observable signals.")

Data contains unhashed raw URLs? False
Data contains unhashed raw client names? False
Privacy Audit Passed: All features are pseudonymized observable signals.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.